In [1]:
!nvidia-smi

Mon Nov 24 15:10:56 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install cuml-cu12 --extra-index-url=https://pypi.nvidia.com

Looking in indexes: https://pypi.org/simple, https://pypi.nvidia.com


In [58]:
import pandas as pd
import numpy as np
import os
import json
import joblib
from google.colab import drive
import ipywidgets as widgets
from IPython.display import display, clear_output
from sklearn.metrics import precision_score

from sklearn.cluster import MiniBatchKMeans
from cuml.ensemble import RandomForestClassifier

In [16]:
import warnings
from sklearn.exceptions import DataConversionWarning

warnings.filterwarnings("ignore", category=UserWarning)  # ignore all UserWarnings
# or more specific
warnings.filterwarnings("ignore", category=DataConversionWarning)

In [4]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [42]:
def _load_kmeans_tree_node(node_path):
    # ---------- Load metadata ----------
    with open(os.path.join(node_path, "node.json"), "r") as f:
        metadata = json.load(f)

    node_id = metadata["node_id"]
    children_ids = metadata["children_ids"]

    # ---------- Load model ----------
    model_path = os.path.join(node_path, "model.joblib")
    model = joblib.load(model_path) if os.path.exists(model_path) else None

    # ---------- Load random forest ----------
    rf_path = os.path.join(node_path, "forest_model/rf_model.joblib")
    if os.path.exists(rf_path):
        rf = joblib.load(rf_path)
    else:
        rf = None

    # ---------- Load dump columns ----------
    rf_zero_var_path = os.path.join(node_path, "forest_model/zero_var_cols.joblib")
    if os.path.exists(rf_zero_var_path):
        zero_var_cols = joblib.load(rf_zero_var_path)
    else:
        zero_var_cols = []

    # ---------- Load children ----------
    children = []
    for cid in children_ids:
        child_path = os.path.join(node_path, cid)
        children.append(_load_kmeans_tree_node(child_path))

    # ---------- Rebuild dictionary ----------
    return {
        "level": metadata["level"],
        "node_id": node_id,
        "n_samples": metadata["n_samples"],
        "k": metadata["k"],
        "kmeans_model": model,
        "children": children,
        "stop_reason": metadata["stop_reason"],
        "rf": rf,
        "zero_var_cols": zero_var_cols,
    }

def load_kmeans_tree(base_path):
    """
    Loads the full KMeans tree saved by save_kmeans_tree().
    Returns the tree dictionary.
    """

    # Find root (only directory at base_path)
    candidates = [d for d in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, d))]
    if len(candidates) != 1:
        raise ValueError("Ambiguous root folder. base_path must contain exactly one folder for the root node.")

    root_id = candidates[0]
    root_path = os.path.join(base_path, root_id)

    return _load_kmeans_tree_node(root_path)

def find_leaf_for_sample(tree, x_row):
    """
    Traverses the tree to find the leaf where this new sample falls.
    x_row: numpy array or single-row DataFrame
    """
    node = tree


    while True:
        # If leaf node → return leaf
        if node["children"] == []:
            x_row = pd.DataFrame(data=x_row, columns=feature_names)
            x_row = x_row.drop(columns=node['zero_var_cols'])
            return node["rf"].predict(x_row)

        model = node["kmeans_model"]
        k = node["k"]

        cluster_id = model.predict(x_row)[0]
        node = node["children"][cluster_id]

In [40]:
# Folder selector
base_path = "/content/drive/Shared drives/Gestió de Projectes/Projecte/saved_model"

tree = load_kmeans_tree(base_path)

In [43]:
# Feature inputs
feature_names = [
    "CONSUMO_REAL",
    "US_AIGUA_SUBM_COMERCIAL",
    "US_AIGUA_SUBM_COMUNITARI",
    "US_AIGUA_SUBM_DOMÈSTIC",
    "US_AIGUA_SUBM_GENERAL",
    "DATE_DIFF",
    "START_MONTH",
    "START_DAY",
    "END_MONTH",
    "END_DAY",
    "Subministrament (€/m³)",
    "Clavegueram (€/m³)",
    "TOTAL (€/m³)",
    "Tarifes socials",
]

# 1. Create a dictionary to hold the feature input widgets
feature_widgets = {}

for f in feature_names:
    # Use FloatText for number input, similar to st.number_input
    feature_widgets[f] = widgets.FloatText(
        value=0.0,
        description=f,
        disabled=False
    )

# 2. Arrange the widgets in two columns using VBox and HBox
column_1_widgets = [feature_widgets[f] for i, f in enumerate(feature_names) if i % 2 == 0]
column_2_widgets = [feature_widgets[f] for i, f in enumerate(feature_names) if i % 2 != 0]

col1 = widgets.VBox(column_1_widgets)
col2 = widgets.VBox(column_2_widgets)

feature_input_box = widgets.HBox([col1, col2])

# 3. Create the Prediction Button and Output area
predict_button = widgets.Button(description="Predict")
output_area = widgets.Output()

# 4. Define the function to run when the button is clicked
def on_predict_button_clicked(b, tree):
    with output_area:
        clear_output(wait=True)
        print("--- Running Prediction ---")

        if not base_path or not os.path.exists(base_path):
            # Display error message
            print(f"ERROR: Invalid base path for the KMeans tree: {base_path}")
            return

        try:
            # Convert to model input
            # Get the current value from each widget
            x_row_data = [feature_widgets[f].value for f in feature_names]
            x_row = np.array([x_row_data], dtype=np.float32)

            # Predict
            prediction = find_leaf_for_sample(tree, x_row)

            # Display success message
            print(f"Predicted CODI_ANOMALIA: {int(prediction[0])}")

        except Exception as e:
            print(f"An unexpected error occurred: {e}")

# 5. Link the button to the prediction function
predict_button.on_click(lambda b: on_predict_button_clicked(b, tree))

# 6. Display all the elements in the Colab notebook
print("KMeans Tree Classifier - Prediction App (Colab Local Display)")
print("Enter the feature values below.")
display(feature_input_box, predict_button, output_area)

KMeans Tree Classifier - Prediction App (Colab Local Display)
Enter the feature values below.


Button(description='Predict', style=ButtonStyle())

Output()

# Precision Test

In [46]:
def load_leaf_data(base_path, node_id):
    """
    Loads (X, y) for a specific leaf node using its node_id.
    """

    # Walk through directories searching for node_id
    for root, dirs, files in os.walk(base_path):
        if os.path.basename(root) == node_id:
            data_path = os.path.join(root, "data.joblib")
            if not os.path.exists(data_path):
                raise ValueError(f"Node {node_id} exists but has no leaf data.")

            return joblib.load(data_path)

    raise ValueError(f"Node {node_id} not found under {base_path}.")



In [61]:
def calculate_precision_on_sample(X, y, tree, feature_names):
    """
    Calculates precision on a sample of data using the provided KMeans tree model.

    Args:
        X (pd.DataFrame): Feature DataFrame.
        y (np.ndarray): True labels array.
        tree (dict): The loaded KMeans tree model.
        feature_names (list): List of feature names.

    Returns:
        float: The calculated precision score, or None if no valid predictions.
    """
    # 1. Select the first 1000 samples
    X_sample = X.head(1000)
    y_sample = y[:1000]

    predictions = []
    for i, row_data in enumerate(X_sample.values):
        x_row = np.array([row_data], dtype=np.float32)
        try:
            # The find_leaf_for_sample expects x_row to be a single row DataFrame internally
            # But the current implementation seems to handle numpy array as input
            # Let's keep it consistent with the previous call which passed np.array.
            prediction = find_leaf_for_sample(tree, x_row)
            predictions.append(int(prediction[0]))
        except Exception as e:
            print(f"Sample {i}: Error during prediction: {e}")
            predictions.append(None)

    # Filter out samples where prediction failed
    valid_predictions = [p for p in predictions if p is not None]
    valid_y_sample = [y_sample[i] for i, p in enumerate(predictions) if p is not None]

    if len(valid_predictions) > 0 and len(valid_y_sample) > 0:
        # Calculate precision
        precision = precision_score(valid_y_sample, valid_predictions, average='weighted', zero_division=0)
        return precision
    else:
        print("No valid predictions to calculate precision.")
        return None

In [62]:
def get_leaf_nodes(node):
    """
    Recursively traverses the tree to find all leaf node IDs.
    """
    leaf_ids = []
    if node["children"] == []:
        leaf_ids.append(node["node_id"])
    else:
        for child in node["children"]:
            leaf_ids.extend(get_leaf_nodes(child))
    return leaf_ids

# 1. Get all leaf node IDs
all_leaf_node_ids = get_leaf_nodes(tree)
print(f"Found {len(all_leaf_node_ids)} leaf nodes.")

# 2. Initialize list to store precision scores
all_precisions = []

# 3. Loop through each leaf_id
for leaf_id in all_leaf_node_ids:
    print(f"\nProcessing leaf node: {leaf_id}")
    try:
        # Load data for the current leaf
        X_leaf, y_leaf = load_leaf_data(base_path, leaf_id)
        print(f"Loaded {len(X_leaf)} samples for leaf {leaf_id}.")

        # Calculate precision
        precision = calculate_precision_on_sample(X_leaf, y_leaf, tree, feature_names)

        # Store and print results
        if precision is not None:
            all_precisions.append({"leaf_id": leaf_id, "precision": precision})
            print(f"Leaf {leaf_id} - Calculated Precision: {precision:.4f}")
        else:
            print(f"Leaf {leaf_id} - Could not calculate precision (no valid predictions).")

    except ValueError as ve:
        print(f"Error loading data for leaf {leaf_id}: {ve}")
    except Exception as e:
        print(f"An unexpected error occurred for leaf {leaf_id}: {e}")

print("\n--- Summary of Leaf Precisions ---")
for result in all_precisions:
    print(f"Leaf ID: {result['leaf_id']}, Precision: {result['precision']:.4f}")

if all_precisions:
    average_precision = np.mean([res['precision'] for res in all_precisions])
    print(f"\nOverall Average Precision across all leaves: {average_precision:.4f}")
else:
    print("No precisions were calculated for any leaf node.")

Found 36 leaf nodes.

Processing leaf node: 39e70ff1-e4ca-4e26-aa38-3ed8f275e0c7
Loaded 561572 samples for leaf 39e70ff1-e4ca-4e26-aa38-3ed8f275e0c7.
Leaf 39e70ff1-e4ca-4e26-aa38-3ed8f275e0c7 - Calculated Precision: 0.8543

Processing leaf node: 5162d10f-0fae-4f5c-a573-d92b56191f42
Loaded 796459 samples for leaf 5162d10f-0fae-4f5c-a573-d92b56191f42.
Leaf 5162d10f-0fae-4f5c-a573-d92b56191f42 - Calculated Precision: 0.8873

Processing leaf node: e76e10bc-4080-417b-92e7-744e84ff8746
Loaded 571059 samples for leaf e76e10bc-4080-417b-92e7-744e84ff8746.
Leaf e76e10bc-4080-417b-92e7-744e84ff8746 - Calculated Precision: 0.7972

Processing leaf node: 3217eb6d-9a09-4d65-ba4a-7d13b1bc5438
Loaded 557098 samples for leaf 3217eb6d-9a09-4d65-ba4a-7d13b1bc5438.
Leaf 3217eb6d-9a09-4d65-ba4a-7d13b1bc5438 - Calculated Precision: 0.8171

Processing leaf node: 1e960e7f-65b4-4fc4-86da-f61b8575512a
Loaded 831165 samples for leaf 1e960e7f-65b4-4fc4-86da-f61b8575512a.
Leaf 1e960e7f-65b4-4fc4-86da-f61b8575512a 